In [7]:
!pip install langchain-tavily

In [1]:
from langchain_classic import hub
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.tools import Tool, tool
#from langchain.tools.retriever import
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch

In [2]:
create_react_agent?

Signature:
create_react_agent(
    llm: 'BaseLanguageModel',
    tools: 'Sequence[BaseTool]',
    prompt: 'BasePromptTemplate',
    output_parser: 'AgentOutputParser | None' = None,
    tools_renderer: 'ToolsRenderer' = <function render_text_description at 0x0000026F3E9BD900>,
    *,
    stop_sequence: 'bool | list[str]' = True,
) -> 'Runnable'
Docstring:
Create an agent that uses ReAct prompting.

Based on paper "ReAct: Synergizing Reasoning and Acting in Language Models"
(https://arxiv.org/abs/2210.03629)

!!! warning

    This implementation is based on the foundational ReAct paper but is older and
    not well-suited for production applications.

    For a more robust and feature-rich implementation, we recommend using the
    `create_agent` function from the `langchain` library.

    See the
    [reference doc](https://reference.langchain.com/python/langchain/agents/)
    for more information.

Args:
    llm: LLM to use as the agent.
    tools: Tools this agent has access to.
    

In [3]:
#To get the groq api
load_dotenv()

True

In [4]:
llm = ChatGroq(model = "meta-llama/llama-4-scout-17b-16e-instruct")

In [5]:
prompt = hub.pull("hwchase17/react")
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [6]:
#Creating an object of Tavily
search_tool = TavilySearch?

Init signature:
TavilySearch(
    *,
    name: str = 'tavily_search',
    description: str = 'A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions about current events. It not only retrieves URLs and snippets, but offers advanced search depths, domain management, time range filters, and image search, this tool delivers real-time, accurate, and citation-backed results.Input should be a search query.',
    args_schema: Type[pydantic.main.BaseModel] = <class 'langchain_tavily.tavily_search.TavilySearchInput'>,
    return_direct: bool = False,
    verbose: bool = False,
    callbacks: list[langchain_core.callbacks.base.BaseCallbackHandler] | langchain_core.callbacks.base.BaseCallbackManager | None = None,
    tags: list[str] | None = None,
    metadata: dict[str, typing.Any] | None = None,
    handle_tool_error: bool = True,
    handle_validation_error: bool | str | collections.abc.Callable[[pydantic_core._pydantic_core.Va

In [7]:
search_tool = TavilySearch()

In [8]:
import datetime
datetime.datetime.now()

datetime.datetime(2026, 4, 27, 12, 51, 53, 363839)

In [9]:
datetime.datetime.now().strftime("%d-%m-%Y %H:%M")

'27-04-2026 12:51'

In [10]:
#A tool that will tell the time by reading it form the system
@tool
def get_time_of_day()->str: # adding a dummy input so that the llm wont give it one.
    """ 
    Returns the current system time.
    only use this when the user explicitly asks for the current time.
    Do not use for general use.
    """
    return datetime.datetime.now().strftime("%d-%m-%Y %H:%M")

In [11]:
get_time_of_day

StructuredTool(name='get_time_of_day', description='Returns the current system time.\nonly use this when the user explicitly asks for the current time.\nDo not use for general use.', args_schema=<class 'langchain_core.utils.pydantic.get_time_of_day'>, func=<function get_time_of_day at 0x0000026F40434700>)

In [12]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

In [13]:
embedding = OpenAIEmbeddings(model = "text-embedding-3-small",dimensions = 512)
db = Chroma(
    persist_directory = (r"C:\Users\chukw\Documents\RAIN\AIML 2ND SEMESTER (File responses)\Rain Class\Week 10 LLM\chromarag.db"),
    embedding_function = embedding
)
retriever = db.as_retriever()

C:\Users\chukw\AppData\Local\Temp\ipykernel_11644\1700854822.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db = Chroma(


In [14]:
retrieval_tool = Tool(name = "retreiver_tool",description="""Use this tool only when you want to get any information about Robotics and artificaial intelligence Nigeria(Rain). Do not use it for anything else. """,func = lambda query: retriever.invoke(query))

In [15]:
react_agent = create_react_agent(llm,tools = [get_time_of_day,search_tool,retrieval_tool],prompt=prompt)

In [16]:
memory = ConversationBufferMemory(return_messsages = True)

C:\Users\chukw\AppData\Local\Temp\ipykernel_11644\2406804896.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(return_messsages = True)


In [17]:
agent = AgentExecutor(agent = react_agent, tools = [get_time_of_day,search_tool,retrieval_tool], handle_parsing_erros = True, verbose = True, memory = memory)

In [18]:
agent.invoke({"input":"What is heavier a kg of steel or a kg of feathers,what is the time?"})



> Entering new AgentExecutor chain...
Thought: The question about what is heavier, a kg of steel or a kg of feathers, is a classic example of a brain teaser. The answer to this part is that they are equal in weight because both weigh 1 kilogram. However, the second part of the question asks for the current time. 

Action: get_time_of_day
Action Input: None27-04-2026 12:52Question: What is heavier a kg of steel or a kg of feathers,what is the time?
Thought: The question about what is heavier, a kg of steel or a kg of feathers, is a classic example of a brain teaser. The answer to this part is that they are equal in weight because both weigh 1 kilogram. However, the second part of the question asks for the current time. 

Action: get_time_of_day
Action Input: None27-04-2026 12:52Thought: I have already obtained the current time, now I need to provide a comprehensive answer to the original question.

Question: What is heavier a kg of steel or a kg of feathers,what is the time?

Thought:

{'input': 'What is heavier a kg of steel or a kg of feathers,what is the time?',
 'history': '',
 'output': 'Agent stopped due to iteration limit or time limit.'}

In [ ]:
agent.invoke({"input":"What is RAIN the company?"})



> Entering new AgentExecutor chain...
Thought: I need to find information about RAIN, which seems to be related to Robotics and Artificial Intelligence in Nigeria. The best tool for this seems to be the retreiver_tool.

Action: retreiver_tool
Action Input: Robotics and artificial intelligence Nigeria(Rain)[Document(metadata={'source': 'C:\\Users\\chukw\\Documents\\RAIN\\AIML 2ND SEMESTER (File responses)\\Rain Class\\Week 10 LLM\\about rain.txt'}, page_content='RAIN HANDLES TRAINING AND CERTIFICATIONS\n RAIN is a research and training facility dedicated to robotics and artificial intelligence technologies. RAIN operates on a standard that is unparalleled in the fast-evolving Robotics and Artificial Intelligence industry and operates out of its research and training facility located in Ibadan, Nigeria. With an average of 30 trainees at every training cohort program, and with a host of highly skilled developers and academics, RAIN presents high competency on a global scale and across d

{'input': 'What is RAIN the company?',
 'history': 'Human: What is heavier a kg of steel or a kg of feathers,what is the time?\nAI: Agent stopped due to iteration limit or time limit.',
 'output': 'RAIN (Robotics and Artificial Intelligence Nigeria) is a research and training facility dedicated to robotics and artificial intelligence technologies. It operates out of its facility in Ibadan, Nigeria, and is known for its high standards in the fast-evolving Robotics and Artificial Intelligence industry. RAIN provides training and certification programs, with an average of 30 trainees per cohort, and is led by Dr. Olusola Ayoola, a visionary leader with a strong academic background in AI, robotics, and engineering. The organization focuses on leveraging cutting-edge technologies to solve critical challenges in various sectors, including healthcare, national security, education, and sustainability. They have also spearheaded multiple groundbreaking projects, including AI-based solutions and

Failed to refresh cache entry hwchase17/react: Connection error caused failure to GET /commits/hwchase17/react/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/hwchase17/react/latest (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.langchain.com\' ([Errno 11001] getaddrinfo failed)"))'))
Content-Length: None
API Key: 
